In [16]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import OneHotEncoder

import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score

#matplotlib 한글

# 사용할 한글 폰트 지정 - 자기 OS만 # 지우고 쓰기
# font_path = "C:/Windows/Fonts/malgun.ttf"  # Windows
font_path = "/System/Library/Fonts/Supplemental/AppleGothic.ttf"  # macOS
# font_path = "/usr/share/fonts/truetype/nanum/NanumGothic.ttf"  # Linux

font_prop = fm.FontProperties(fname=font_path)

# 폰트 적용
plt.rc('font', family=font_prop.get_name())

train = pd.read_csv('csv/train.csv')
test = pd.read_csv('csv/test.csv')
sample_submission = pd.read_csv('csv/sample_submission.csv')

#특성과 타겟 변수 분리
train = train.drop(columns=['ID'], axis = 1)
test = test.drop(columns=['ID'], axis = 1)


In [17]:
np.random.seed(42)
train['성공여부'] = np.random.binomial(1, train['성공확률'])

In [18]:
# 설립연도 -> 2025-설립연도
train['설립연도'] =2025- train['설립연도']
test['설립연도'] =2025- test['설립연도']

# 기업가치 매핑
ev_map={'1500-2500':2,'2500-3500':3,'3500-4500':4,'4500-6000':5.25,'6000이상':6.5}
train['기업가치(백억원)']=train['기업가치(백억원)'].map(ev_map)
test['기업가치(백억원)']=test['기업가치(백억원)'].map(ev_map)
# 투자단계 매핑
round_map={'Seed':1,'Series A':2, 'Series B':3, 'Series C':4, 'IPO':5}
train['투자단계']=train['투자단계'].map(round_map)
test['투자단계']=test['투자단계'].map(round_map)

category_features = ['국가','분야','투자단계']
numeric_features = ['설립연도','직원 수','고객수(백만명)','총 투자금(억원)','연매출(억원)','SNS 팔로워 수(백만명)','기업가치(백억원)']
bool_features = ['인수여부','상장여부']


# 수치형 변수 결측치를 평균값으로 대체
for feature in numeric_features:
    mean_value = train[feature].mean()
    train[feature] = train[feature].fillna(mean_value)
    test[feature] = test[feature].fillna(mean_value)
    


In [19]:
# OneHotEncoder 객체를 각 범주형 feature별로 따로 저장하여 사용
encoders = {}

# 범주형 데이터를 encoding
for feature in ['국가','분야']:
    encoders[feature] = OneHotEncoder(sparse_output=False)
    train[feature] = train[feature].fillna('Missing')
    test[feature] = test[feature].fillna('Missing')
    
    encoded = encoders[feature].fit_transform(train[[feature]])
    encoded_df = pd.DataFrame(encoded, columns=encoders[feature].get_feature_names_out([feature]))
    train=pd.concat([train.drop(columns=feature).reset_index(drop=True),encoded_df],axis=1)
    
    encoded = encoders[feature].transform(test[[feature]])
    encoded_df = pd.DataFrame(encoded, columns=encoders[feature].get_feature_names_out([feature]))
    test=pd.concat([test.drop(columns=feature).reset_index(drop=True),encoded_df],axis=1)

# 불리언 값을 0과 1로 변환 ('Yes' → 1, 'No' → 0 으로 변환)
bool_map = {'Yes': 1, 'No': 0}

for feature in bool_features:
    train[feature] = train[feature].map(bool_map)
    test[feature] = test[feature].map(bool_map)



In [20]:
numeric_features=['설립연도','직원 수','고객수(백만명)','총 투자금(억원)','연매출(억원)','SNS 팔로워 수(백만명)','투자단계','기업가치(백억원)']
bool_features = ['인수여부','상장여부']
onehot_features=list(set(train.columns)-set(numeric_features)-set(bool_features)-{'성공확률','성공여부'})

train[onehot_features]=train[onehot_features].astype(int)
test[onehot_features]=test[onehot_features].astype(int)

In [21]:
#log 변환
for feature in ['총 투자금(억원)','연매출(억원)','직원 수']:
    train[feature]=np.log10(train[feature]+1)
    train.rename(columns={feature:'log'+feature})
train['SNS 팔로워 수(백만명)']=np.log10(train['SNS 팔로워 수(백만명)']+0.01)
train.rename(columns={'SNS 팔로워 수(백만명)':'log SNS 팔로워 수(백만명)'})
train['기업가치(백억원)']=np.log10(train['기업가치(백억원)'])
train.rename(columns={'기업가치(백억원)':'log 기업가치(백억원)'})

,설립연도,투자단계,직원 수,인수여부,상장여부,고객수(백만명),총 투자금(억원),연매출(억원),SNS 팔로워 수(백만명),log 기업가치(백억원),...,분야_Missing,분야_게임,분야_기술,분야_물류,분야_에너지,분야_에듀테크,분야_이커머스,분야_푸드테크,분야_핀테크,분야_헬스케어
0,16,2,3.615634,0,0,56.000000,3.527114,3.678063,0.673942,0.617413,...,0,0,0,0,0,0,1,0,0,0
1,2,1,3.619928,1,0,80.000000,3.609594,2.447158,0.004321,0.477121,...,0,0,0,0,0,0,0,0,1,0
2,7,2,3.495960,1,1,54.000000,3.809829,4.084290,0.603144,0.602060,...,0,0,1,0,0,0,0,0,0,0
3,9,1,3.511349,1,1,49.214332,2.823474,4.023170,0.474216,0.617413,...,1,0,0,0,0,0,0,0,0,0
4,5,1,3.294466,0,1,94.000000,2.919078,3.991713,0.004321,0.301030,...,0,0,0,0,0,1,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4371,4,2,3.685025,1,0,90.000000,3.622007,3.972897,0.603144,0.301030,...,0,0,0,0,0,0,0,1,0,0
4372,5,3,2.745075,0,1,37.000000,2.901458,3.472756,0.478566,0.812913,...,0,0,0,0,1,0,0,0,0,0
4373,2,4,2.705008,0,1,49.214332,3.520484,3.654465,0.170262,0.617413,...,1,0,0,0,0,0,0,0,0,0
4374,24,5,3.158061,0,0,53.000000,3.379487,3.574726,0.699838,0.720159,...,0,0,0,0,0,0,0,0,0,0


In [22]:
#log 변환
for feature in ['총 투자금(억원)','연매출(억원)','직원 수']:
    test[feature]=np.log10(test[feature]+1)
    test.rename(columns={feature:'log'+feature})
test['SNS 팔로워 수(백만명)']=np.log10(test['SNS 팔로워 수(백만명)']+0.01)
test.rename(columns={'SNS 팔로워 수(백만명)':'log SNS 팔로워 수(백만명)'})
test['기업가치(백억원)']=np.log10(test['기업가치(백억원)'])
test.rename(columns={'기업가치(백억원)':'log 기업가치(백억원)'})


,설립연도,투자단계,직원 수,인수여부,상장여부,고객수(백만명),총 투자금(억원),연매출(억원),SNS 팔로워 수(백만명),log 기업가치(백억원),...,분야_Missing,분야_게임,분야_기술,분야_물류,분야_에너지,분야_에듀테크,분야_이커머스,분야_푸드테크,분야_핀테크,분야_헬스케어
0,23,4,3.513484,0,1,45.0,3.700877,3.824841,0.303196,0.301030,...,0,0,0,0,0,0,0,0,1,0
1,5,4,3.569140,1,0,70.0,3.204663,3.667920,0.624282,0.617413,...,0,0,0,0,0,0,0,1,0,0
2,11,5,2.374748,1,1,89.0,3.673021,3.968016,0.004321,0.812913,...,0,0,0,0,0,1,0,0,0,0
3,22,1,2.804821,1,1,17.0,3.331630,3.845470,0.699838,0.301030,...,0,0,0,0,1,0,0,0,0,0
4,19,1,3.692230,1,0,68.0,3.698622,3.880471,0.640481,0.617413,...,0,0,0,0,0,0,0,0,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1750,10,1,3.458033,1,1,49.0,2.732394,3.288473,0.603144,0.301030,...,0,0,0,0,0,1,0,0,0,0
1751,19,5,2.445604,1,1,35.0,3.375481,4.035350,0.478566,0.720159,...,0,0,0,0,0,0,0,0,1,0
1752,23,5,3.169968,0,1,96.0,3.624901,3.918973,0.478566,0.301030,...,0,0,1,0,0,0,0,0,0,0
1753,3,5,3.552790,0,0,59.0,3.522966,3.146128,0.699838,0.720159,...,0,0,0,0,0,0,0,0,1,0


In [23]:
#A&B
temp_dict={}
temp_dict['인수&상장']=train['인수여부']*train['상장여부']
train = pd.concat([train, pd.DataFrame(temp_dict)], axis=1)

In [24]:
#A&B
temp_dict={}
temp_dict['인수&상장']=test['인수여부']*test['상장여부']
test = pd.concat([test, pd.DataFrame(temp_dict)], axis=1)

In [25]:
# 수치형 간 비율 - 직원 수 당, 설립연도 당, 투자금 당, 투자단계 당 - (input으로 나눠주기)
output_list=['고객수(백만명)','연매출(억원)','SNS 팔로워 수(백만명)','기업가치(백억원)']
input_list_1=['직원 수','총 투자금(억원)']
input_list_2=['설립연도','투자단계']

temp_dict={}
for x in input_list_1+input_list_2:
    for y in output_list:
        temp_dict[y+'/'+x]=train[y]/train[x]
        numeric_features.append(y+'/'+x)
for x in input_list_2:
    for y in input_list_1:
        temp_dict[y+'/'+x]=train[y]/train[x]
        numeric_features.append(y+'/'+x)

train = pd.concat([train, pd.DataFrame(temp_dict)], axis=1)

In [26]:
# 수치형 간 비율 - 직원 수 당, 설립연도 당, 투자금 당, 투자단계 당 - (input으로 나눠주기)
output_list=['고객수(백만명)','연매출(억원)','SNS 팔로워 수(백만명)','기업가치(백억원)']
input_list_1=['직원 수','총 투자금(억원)']
input_list_2=['설립연도','투자단계']

temp_dict={}
for x in input_list_1+input_list_2:
    for y in output_list:
        temp_dict[y+'/'+x]=test[y]/test[x]
        numeric_features.append(y+'/'+x)
for x in input_list_2:
    for y in input_list_1:
        temp_dict[y+'/'+x]=test[y]/test[x]
        numeric_features.append(y+'/'+x)

test = pd.concat([test, pd.DataFrame(temp_dict)], axis=1)

In [27]:
X=train.drop(columns=['성공확률','성공여부'],axis=1)
y_prob=train['성공확률']
y_label=train['성공여부']
X_train, X_test, y_prob_train, y_prob_test, y_label_train, y_label_test = train_test_split(X, y_prob, y_label, test_size=0.2, random_state=42)

In [ ]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression()
model.fit(X_train, y_label_train)


train_probs = model.predict_proba(X_train)[:,1]
train_prob_score = r2_score(y_prob_train, train_probs)
print("Best R² on train prob set:", train_prob_score)

train_labels = model.predict(X_train)
train_label_score = r2_score(y_label_train, train_labels)
print("Best R² on train label set:", train_label_score)


probs = model.predict_proba(X_test)[:,1]
prob_score = r2_score(y_prob_test, probs)
print("\nBest R² on test prob set:", prob_score)

labels = model.predict(X_test)
label_score = r2_score(y_label_test, labels)
print("Best R² on test label set:", label_score)



Best R² on train prob set: -0.01060529095765661
Best R² on train label set: -0.8508440170204163
Best R² on test prob set: -0.04432493073069432
Best R² on test label set: -0.9935277783577263


/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [60]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(n_estimators=3000, max_depth=7)
model.fit(X_train, y_label_train)


train_probs = model.predict_proba(X_train)[:,1]
train_prob_score = r2_score(y_prob_train, train_probs)
print("Best R² on train prob set:", train_prob_score)

train_labels = model.predict(X_train)
train_label_score = r2_score(y_label_train, train_labels)
print("Best R² on train label set:", train_label_score)


probs = model.predict_proba(X_test)[:,1]
prob_score = r2_score(y_prob_test, probs)
print("\nBest R² on test prob set:", prob_score)

labels = model.predict(X_test)
label_score = r2_score(y_label_test, labels)
print("Best R² on test label set:", label_score)


Best R² on train prob set: 0.11552411436120191
Best R² on train label set: 0.09065231918103611

Best R² on test prob set: -0.035310391883262904
Best R² on test label set: -0.915798484247448


In [ ]:
from sklearn.ensemble import GradientBoostingClassifier

model = GradientBoostingClassifier(
    n_estimators=500,
    learning_rate=0.005,
    max_depth=7,
    max_features='log2'
    )
model.fit(X_train, y_label_train)


train_probs = model.predict_proba(X_train)[:,1]
train_prob_score = r2_score(y_prob_train, train_probs)
print("Best R² on train prob set:", train_prob_score)

train_labels = model.predict(X_train)
train_label_score = r2_score(y_label_train, train_labels)
print("Best R² on train label set:", train_label_score)


probs = model.predict_proba(X_test)[:,1]
prob_score = r2_score(y_prob_test, probs)
print("\nBest R² on test prob set:", prob_score)

labels = model.predict(X_test)
label_score = r2_score(y_label_test, labels)
print("Best R² on test label set:", label_score)

Best R² on train prob set: 0.14511042557833898
Best R² on train label set: 0.5166220029990104

Best R² on test prob set: -0.07697956742087197
Best R² on test label set: -0.9340877299204546


In [68]:
from xgboost import XGBClassifier

model = XGBClassifier(
    learning_rate=0.005,
    n_estimators=1000,
    max_depth=None,
    subsample=0.8,
    eval_metric='logloss'
    )
model.fit(X_train, y_label_train)


train_probs = model.predict_proba(X_train)[:,1]
train_prob_score = r2_score(y_prob_train, train_probs)
print("Best R² on train prob set:", train_prob_score)

train_labels = model.predict(X_train)
train_label_score = r2_score(y_label_train, train_labels)
print("Best R² on train label set:", train_label_score)


probs = model.predict_proba(X_test)[:,1]
prob_score = r2_score(y_prob_test, probs)
print("\nBest R² on test prob set:", prob_score)

labels = model.predict(X_test)
label_score = r2_score(y_label_test, labels)
print("Best R² on test label set:", label_score)

Best R² on train prob set: 0.1667162035129255
Best R² on train label set: 0.7761075785862399

Best R² on test prob set: -0.15808131535936942
Best R² on test label set: -1.0163893354489848


In [40]:
from sklearn.neighbors import KNeighborsClassifier

model = KNeighborsClassifier(n_neighbors=5)
model.fit(X_train, y_label_train)


train_probs = model.predict_proba(X_train)[:,1]
train_prob_score = r2_score(y_prob_train, train_probs)
print("Best R² on train prob set:", train_prob_score)

train_labels = model.predict(X_train)
train_label_score = r2_score(y_label_train, train_labels)
print("Best R² on train label set:", train_label_score)


probs = model.predict_proba(X_test)[:,1]
prob_score = r2_score(y_prob_test, probs)
print("\nBest R² on test prob set:", prob_score)

labels = model.predict(X_test)
label_score = r2_score(y_label_test, labels)
print("Best R² on test label set:", label_score)

Best R² on train prob set: -0.46083464463388757
Best R² on train label set: -0.29053888035418596

Best R² on test prob set: -0.8987284746769333
Best R² on test label set: -1.1535586779965348


In [ ]:
'''from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

# 회귀 모델 예시
model = GradientBoostingRegressor(
    n_estimators=100,
    learning_rate=0.005,
    max_depth=None,
    subsample=1,
    max_features='log2',
    random_state=42
)
model.fit(X, y_prob)

'''

GradientBoostingRegressor(learning_rate=0.005, max_depth=None,
                          max_features='log2', random_state=42, subsample=1)

In [ ]:
'''final_predictions = model.predict(test)

sample_submission['성공확률'] = final_predictions
sample_submission.to_csv('./submission3.csv', index = False, encoding = 'utf-8-sig')'''